# External feature linking -- new LINZ-matched coordinates, standalone

`dvrs_LINZ_matched.csv` already carries its own join key
(`valuation_id_concat`) and coordinates (`shape_X`/`shape_Y`), so this
notebook works from it alone -- it does not load or merge with
`cache/dvrs_processed.parquet`. This notebook's only job is linking
external GIS features onto these coordinates; joining the result back onto
the full sales table (and deciding what to do with the ~18.3% of sales this
file has no match for) is left to whoever consumes its output.

Output: `cache/external_features_linz.parquet`, keyed by
`valuation_id_concat`, one row per matched property (81,363 rows -- every
row in the source file has a coordinate, so there's nothing to filter here).

## 1. Load

In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

linz = pd.read_csv("./dvrs_LINZ_matched.csv")
print(f"{len(linz):,} matched properties, {linz['valuation_id_concat'].nunique():,} unique keys")

linz = linz.rename(columns={"shape_X": "longitude", "shape_Y": "latitude"})
print(linz[["valuation_id_concat", "longitude", "latitude"]].head())

81,363 matched properties, 81,363 unique keys
  valuation_id_concat   longitude   latitude
0        1_2501710000  174.706389 -36.865688
1        1_2600010100  174.708779 -36.869417
2        1_4300000201  174.702890 -36.861304
3        1_4300110000  174.700846 -36.862339
4        1_4300270001  174.698847 -36.863616


## 2. Link external GIS features

Same layers and spatial-join logic as `dvrp270826.ipynb` (local static
files in `hazard_data/`): flood plains, flood sensitive areas (boolean +
nearest distance), Unitary Plan base zone.

In [2]:
flood_plains = gpd.read_file("./hazard_data/flood_plains.geojson")
flood_sensitive = gpd.read_file("./hazard_data/flood_sensitive_area.geojson")
unitary_plan_zones = gpd.read_file("./hazard_data/Unitary_Plan_Base_Zone.geojson")

point_gdf = gpd.GeoDataFrame(
    linz[[]],
    geometry=gpd.points_from_xy(linz["longitude"], linz["latitude"]),
    crs="EPSG:4326",
)
print(f"{len(point_gdf):,} points to link")

81,363 points to link


In [3]:
flood_plains_matches = gpd.sjoin(point_gdf, flood_plains, how="inner", predicate="within")
linz["in_flood_plain"] = linz.index.isin(flood_plains_matches.index)

flood_sensitive_matches = gpd.sjoin(point_gdf, flood_sensitive, how="inner", predicate="within")
linz["in_flood_sensitive_area"] = linz.index.isin(flood_sensitive_matches.index)

unitary_zone = gpd.sjoin(point_gdf, unitary_plan_zones, how="left", predicate="within")
unitary_zone = unitary_zone[~unitary_zone.index.duplicated(keep="first")]
linz["unitary_plan_zone"] = unitary_zone["ZONE"]

print(linz[["in_flood_plain", "in_flood_sensitive_area"]].mean())
print()
print(f"unitary_plan_zone NaN (point outside every mapped zone): {linz['unitary_plan_zone'].isna().sum():,}")

in_flood_plain             0.076865
in_flood_sensitive_area    0.027863
dtype: float64

unitary_plan_zone NaN (point outside every mapped zone): 3


In [4]:
points_nztm = point_gdf.to_crs("EPSG:2193")
flood_plains_nztm = flood_plains.to_crs("EPSG:2193")
flood_sensitive_nztm = flood_sensitive.to_crs("EPSG:2193")

flood_plains_nearest = gpd.sjoin_nearest(points_nztm, flood_plains_nztm, distance_col="dist_to_flood_plain_m")
flood_plains_nearest = flood_plains_nearest[~flood_plains_nearest.index.duplicated(keep="first")]
linz["dist_to_flood_plain_m"] = flood_plains_nearest["dist_to_flood_plain_m"]

flood_sensitive_nearest = gpd.sjoin_nearest(points_nztm, flood_sensitive_nztm, distance_col="dist_to_flood_sensitive_area_m")
flood_sensitive_nearest = flood_sensitive_nearest[~flood_sensitive_nearest.index.duplicated(keep="first")]
linz["dist_to_flood_sensitive_area_m"] = flood_sensitive_nearest["dist_to_flood_sensitive_area_m"]

print(linz[["dist_to_flood_plain_m", "dist_to_flood_sensitive_area_m"]].describe())

       dist_to_flood_plain_m  dist_to_flood_sensitive_area_m
count           81363.000000                    81363.000000
mean               93.185814                     2675.547041
std                77.382434                     5213.405223
min                 0.000000                        0.000000
25%                30.841064                      217.640816
50%                78.538661                     1245.456868
75%               139.872714                     3075.209974
max              1068.241175                    80194.468722


## 3. Save

In [5]:
OUT_PATH = Path("./cache/external_features_linz.parquet")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

out_cols = [
    "valuation_id_concat", "longitude", "latitude",
    "in_flood_plain", "in_flood_sensitive_area",
    "dist_to_flood_plain_m", "dist_to_flood_sensitive_area_m",
    "unitary_plan_zone",
]
linz[out_cols].to_parquet(OUT_PATH)

print(f"Saved {linz[out_cols].shape} to {OUT_PATH}")

Saved (81363, 8) to cache\external_features_linz.parquet


## 4. Join back onto the full sales/property data

`dvrs_LINZ_matched.csv` has coordinates only, no property attributes or
sale price -- those come from the raw Auckland Council sales file
`DVRS270826.txt`, parsed directly (same logic as `dvrp270826.ipynb`:
UTF-16, pipe-delimited, quoted lines unwrapped, malformed rows dropped),
rather than from the already-processed `cache/dvrs_processed.parquet`.

**Join key:** `valuation_id_concat` must include `valuation_number_suffix`
when present -- this distinguishes sub-units (cross-leases, unit titles)
that share one `valuation_roll_number`/`valuation_number_assessment`. The
suffix is also lowercased, since the new file uses lowercase (`"d"`) where
the base data uses uppercase (`"D"`). Omitting the suffix, or not matching
its case, silently collapses distinct properties onto the same key and
undercounts matches -- this was caught by checking why only 74,143/90,738
rows matched when it should have been far higher.

In [6]:
import csv
import io

with open("DVRS270826.txt", encoding="utf-16") as f:
    lines = f.read().splitlines()

header, *rows = lines
expected_cols = len(header.split("|"))


def unwrap(line):
    if len(line) >= 2 and line[0] == '"' and line[-1] == '"':
        return line[1:-1]
    return line


cleaned = [unwrap(line) for line in rows]
good_rows = [line for line in cleaned if len(line.split("|")) == expected_cols]
n_dropped = len(cleaned) - len(good_rows)
if n_dropped:
    print(f"Dropping {n_dropped} malformed row(s) that don't split into {expected_cols} fields")

csv_text = "\n".join([header] + good_rows)
base = pd.read_csv(io.StringIO(csv_text), sep="|", dtype=str, quoting=csv.QUOTE_NONE)

base.columns = (
    base.columns.str.strip().str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")
)
print(f"Parsed {base.shape} from DVRS270826.txt")

Dropping 1 malformed row(s) that don't split into 41 fields

Parsed (92010, 41) from DVRS270826.txt


In [7]:
def build_valuation_id(row):
    suffix = row["valuation_number_suffix"]
    key = f"{row['valuation_roll_number']}_{row['valuation_number_assessment']}"
    if pd.notna(suffix) and str(suffix).strip() != "":
        key += f"_{str(suffix).strip().lower()}"
    return key


base["valuation_id_concat"] = base.apply(build_valuation_id, axis=1)
print(f"Unique keys: {base['valuation_id_concat'].nunique():,} / {len(base):,} rows "
      f"({base['valuation_id_concat'].duplicated().sum():,} duplicate keys -- "
      f"these are repeat sales of the same property across 2024-2026, not an error: "
      f"a left join still attaches the right coordinate to every sale)")

external = pd.read_parquet("./cache/external_features_linz.parquet")

combined = base.merge(external, on="valuation_id_concat", how="left")
combined["linz_matched"] = combined["longitude"].notna()

print(f"Combined: {combined.shape}")
print(f"linz_matched: {combined['linz_matched'].sum():,} / {len(combined):,} "
      f"({combined['linz_matched'].mean():.1%})")
print()
print(combined[["longitude", "latitude", "in_flood_plain", "unitary_plan_zone", "linz_matched"]].head())

Unique keys: 85,180 / 92,010 rows (6,830 duplicate keys -- these are repeat sales of the same property across 2024-2026, not an error: a left join still attaches the right coordinate to every sale)
Combined: (92010, 50)
linz_matched: 87,916 / 92,010 (95.6%)

    longitude   latitude in_flood_plain  unitary_plan_zone  linz_matched
0  174.706389 -36.865688          False               60.0          True
1  174.708779 -36.869417          False                8.0          True
2  174.702890 -36.861304          False               60.0          True
3  174.700846 -36.862339          False               18.0          True
4  174.698847 -36.863616          False               18.0          True


## 5. Save the combined dataset

In [8]:
COMBINED_PATH = Path("./cache/dvrs_processed_linz_matched.parquet")
combined.to_parquet(COMBINED_PATH)
print(f"Saved {combined.shape} to {COMBINED_PATH}")

Saved (92010, 50) to cache\dvrs_processed_linz_matched.parquet


## 6. Summary

- Property/sale data now comes directly from `DVRS270826.txt` (parsed
  fresh, same logic as `dvrp270826.ipynb`), not from
  `cache/dvrs_processed.parquet` -- this notebook no longer depends on that
  cached artifact at all.
- **Join-key fix:** `valuation_id_concat` must include
  `valuation_number_suffix` (lowercased) to correctly distinguish
  sub-units. Without it, the match rate was silently undercounted at
  81.7% (74,143/90,738); with it, it rises to the figures reported below.
- `external_features_linz.parquet` (coordinates + GIS features from the new
  LINZ linkage) is joined on this corrected key.
- `linz_matched` is the data-quality flag (replacing `street_in_linz`) --
  rows where it's `False` have nulls for every coordinate/GIS column.
- Saved to `cache/dvrs_processed_linz_matched.parquet`.
- **Still not done:** re-joining the team's noise/consents/travel-distance
  features onto these new coordinates, and re-running `final.ipynb`'s
  ablation on the result.